Reminder, make sure you've built the package:
```bash
maturin develop --release
```

In [ ]:
# Import KADAR modules
from kadar import (
    GenomeIslandPredictor,
    KmerProfiler,
    SlidingWindowScanner,
    IVOMAnalysis,
    load_fasta,
    load_fasta_string,
    build_kmer_profile,
)

import matplotlib.pyplot as plt
import numpy as np

print("KADAR imported successfully!")

In [ ]:
import random

def generate_sequence(length, gc_content=0.5):
    """Generate a random DNA sequence with specified GC content."""
    gc_count = int(length * gc_content)
    at_count = length - gc_count
    
    bases = (
        ['G'] * (gc_count // 2) + 
        ['C'] * (gc_count // 2) + 
        ['A'] * (at_count // 2) + 
        ['T'] * (at_count // 2)
    )
    # Pad if odd
    while len(bases) < length:
        bases.append(random.choice(['A', 'T', 'G', 'C']))
    
    random.shuffle(bases)
    return ''.join(bases)

# Generate a "host" genome with ~50% GC
random.seed(42)
host_gc = 0.50
genome_length = 100000  # 100kb

# Create genome in three parts: before island, island, after island
island_start = 40000
island_length = 15000  # 15kb island
island_gc = 0.35  # Island has different GC content

part1 = generate_sequence(island_start, gc_content=host_gc)
island_seq = generate_sequence(island_length, gc_content=island_gc)
part2 = generate_sequence(genome_length - island_start - island_length, gc_content=host_gc)

synthetic_genome = part1 + island_seq + part2

print(f"Generated synthetic genome: {len(synthetic_genome)} bp")
print(f"Host GC content: {host_gc:.0%}")
print(f"Island location: {island_start}-{island_start + island_length}")
print(f"Island GC content: {island_gc:.0%}")

In [ ]:
predictor = GenomeIslandPredictor(
    k=10,                    # 4-mer analysis
    window_size=1000,       # 5kb windows
    step_size=100,         # 1kb step
    min_island_size=8000,   # minimum 8kb islands
    score_threshold=0.05    # sensitivity threshold
)

# Predict islands
islands = predictor.predict(synthetic_genome, "synthetic_genome")

print(f"\nFound {len(islands)} genomic island(s):\n")
for i, island in enumerate(islands, 1):
    print(f"Island {i}:")
    print(f"  Location: {island.start:,} - {island.end:,} bp")
    print(f"  Length: {island.length():,} bp")
    print(f"  Score: {island.score:.4f}")
    print(f"  GC content: {island.gc_content:.2%}")
    print(f"  K-mer deviation: {island.kmer_deviation:.4f}")
    print()

In [ ]:
# Get detailed window scores
islands, scores = predictor.predict_with_scores(synthetic_genome, "synthetic_genome")

# Extract data for plotting
positions = [s.position / 1000 for s in scores]  # Convert to kb
deviations = [s.kmer_deviation for s in scores]
gc_values = [s.gc_content for s in scores]
total_scores = [s.score for s in scores]

# Create figure
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Plot 1: K-mer deviation
ax1 = axes[0]
ax1.plot(positions, deviations, 'b-', linewidth=0.8)
ax1.axhline(y=predictor.score_threshold, color='r', linestyle='--', 
            label=f'Threshold ({predictor.score_threshold})')
ax1.axvspan(island_start/1000, (island_start + island_length)/1000, 
            alpha=0.2, color='green', label='True island')
ax1.set_ylabel('K-mer Deviation')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Plot 2: GC content
ax2 = axes[1]
ax2.plot(positions, gc_values, 'g-', linewidth=0.8)
ax2.axhline(y=host_gc, color='gray', linestyle=':', label=f'Host GC ({host_gc:.0%})')
ax2.axvspan(island_start/1000, (island_start + island_length)/1000, 
            alpha=0.2, color='green')
ax2.set_ylabel('GC Content')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

# Plot 3: Combined score
ax3 = axes[2]
ax3.fill_between(positions, total_scores, alpha=0.5, color='purple')
ax3.plot(positions, total_scores, 'purple', linewidth=0.8)
ax3.axhline(y=predictor.score_threshold, color='r', linestyle='--')
ax3.axvspan(island_start/1000, (island_start + island_length)/1000, 
            alpha=0.2, color='green')

# Mark detected islands
for island in islands:
    ax3.axvspan(island.start/1000, island.end/1000, 
                alpha=0.3, color='red', label='Detected')

ax3.set_ylabel('Combined Score')
ax3.set_xlabel('Position (kb)')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create a k-mer profiler
profiler = KmerProfiler(k=10, normalize=True)

# Add the host region and island region separately
host_region = part1 + part2
profiler.add_sequence("host", host_region)
profiler.add_sequence("island", island_seq)

# Compare the sequences
similarity = profiler.jaccard_similarity("host", "island")
divergence = profiler.js_divergence("host", "island")

print(f"Comparison between host and island regions:")
print(f"  Jaccard similarity: {similarity:.4f}")
print(f"  Jensen-Shannon divergence: {divergence:.4f}")

In [ ]:
# Get k-mer frequencies and visualize top k-mers
host_counts = profiler.get_kmer_counts("host")
island_counts = profiler.get_kmer_counts("island")

# Find k-mers with largest frequency differences
all_kmers = set(host_counts.keys()) | set(island_counts.keys())
differences = []
for kmer in all_kmers:
    host_freq = host_counts.get(kmer, 0)
    island_freq = island_counts.get(kmer, 0)
    diff = island_freq - host_freq
    differences.append((kmer, host_freq, island_freq, diff))

# Sort by absolute difference
differences.sort(key=lambda x: abs(x[3]), reverse=True)

print("Top 10 k-mers with largest frequency differences:")
print(f"{'K-mer':<10} {'Host Freq':>12} {'Island Freq':>12} {'Difference':>12}")
print("-" * 50)
for kmer, host_f, island_f, diff in differences[:10]:
    print(f"{kmer:<10} {host_f:>12.4f} {island_f:>12.4f} {diff:>+12.4f}")

In [ ]:
# Use the sliding window scanner directly
scanner = SlidingWindowScanner(
    window_size=2000,  # Smaller windows for finer resolution
    step_size=500,
    k=10
)

# Scan the genome
window_scores = scanner.scan(synthetic_genome)

print(f"Scanned {len(window_scores)} windows")
print(f"\nFirst n windows:")
for score in window_scores[:10]:
    print(f"  Position {score.position:>6}: deviation={score.kmer_deviation:.4f}, GC={score.gc_content:.2%}")

In [ ]:
# Create IVOM analyzer
ivom = IVOMAnalysis(max_order=10, min_order=1)

# Build background model from host regions
ivom.build_background_model([host_region])

# Calculate deviation for different regions
host_sample = part1[:10000]  # Sample from host
island_sample = island_seq[:10000]  # Sample from island

host_deviation = ivom.calculate_deviation(host_sample)
island_deviation = ivom.calculate_deviation(island_sample)

print(f"IVOM deviation scores:")
print(f"  Host region sample: {host_deviation:.4f}")
print(f"  Island region sample: {island_deviation:.4f}")
print(f"\nThe island shows {island_deviation/host_deviation:.1f}x higher deviation.")

In [ ]:
# View IVOM interpolation weights
weights = ivom.get_weights()
orders = list(range(ivom.min_order, ivom.max_order + 1))

plt.figure(figsize=(8, 4))
plt.bar(orders, weights, color='steelblue', edgecolor='black')
plt.xlabel('K-mer Order')
plt.ylabel('Weight')
plt.title('IVOM Interpolation Weights')
plt.xticks(orders)
plt.grid(True, alpha=0.3, axis='y')
plt.show()

In [ ]:
# Test different threshold values
thresholds = [0.02, 0.05, 0.08, 0.10, 0.15]

print(f"Effect of score threshold on island detection:")
print(f"(True island: {island_start:,}-{island_start + island_length:,})\n")
print(f"{'Threshold':>10} {'# Islands':>10} {'Detected Range':>25}")
print("-" * 50)

for threshold in thresholds:
    predictor.score_threshold_py = threshold
    islands = predictor.predict(synthetic_genome, "test")
    
    if islands:
        ranges = ", ".join([f"{i.start//1000:.0f}-{i.end//1000:.0f}kb" for i in islands])
    else:
        ranges = "None"
    
    print(f"{threshold:>10.2f} {len(islands):>10} {ranges:>25}")

In [ ]:
# Test different k-mer sizes
k_values = [3, 4, 5, 6, 7, 8, 9]

print(f"\nEffect of k-mer size on detection:")
print(f"{'K-mer size':>12} {'# Islands':>10} {'Avg Score':>12}")
print("-" * 40)

for k in k_values:
    pred = GenomeIslandPredictor(k=k, score_threshold=0.05)
    islands = pred.predict(synthetic_genome, "test")
    
    avg_score = np.mean([i.score for i in islands]) if islands else 0
    print(f"{k:>12} {len(islands):>10} {avg_score:>12.4f}")